# Week 9 — Decision Trees, Random Forests, and Feature Importance

**ESC 113: Computer Aided Analysis for Engineering**

This notebook is designed for a **3-hour class**. Run each code cell, read the explanations, complete the guided exercises, then finish the quiz without looking at the worked examples.

## Table of Contents

1. Decision Tree Fundamentals
2. Tree Splits and Maximum Depth
3. Overfitting
4. Random Forests
5. Ensemble Learning
6. Loading the UCI Concrete Compressive Strength Dataset
7. Decision Tree Regression
8. Random Forest Regression
9. Model Performance Comparison
10. Training vs Test Performance
11. Feature Importance
12. Engineering Interpretation
13. Open-Data Exercises
14. Engineering Discussion Problems
15. Review and Quiz

### Dataset
UCI Concrete Compressive Strength Dataset

In [ ]:
!pip -q install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

concrete = fetch_ucirepo(id=165)
X = concrete.data.features.copy()
y = concrete.data.targets.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Decision tree

In [ ]:
tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
p_tree = tree.predict(X_test)

## 2. Random forest

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
p_rf = rf.predict(X_test)

## 3. Compare models

In [ ]:
def metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred)
    }

results = pd.DataFrame({
    "Decision Tree": metrics(y_test, p_tree),
    "Random Forest": metrics(y_test, p_rf)
}).T
results

## 4. Overfitting demonstration

Compare training and testing R² for trees of different depths.

In [ ]:
rows = []
for depth in [2, 4, 6, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "train_R2": m.score(X_train, y_train),
        "test_R2": m.score(X_test, y_test)
    })
pd.DataFrame(rows)

## 5. Feature importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7,5))
plt.barh(imp.index, imp.values)
plt.xlabel("Random Forest Feature Importance")
plt.title("Concrete Strength: Feature Importance")
plt.tight_layout()
plt.show()

### Interpretation caution

Feature importance describes how the fitted model uses variables; it does not prove physical causation. Correlated inputs can also redistribute importance.

## 6. Open-data exercises

1. Identify the three most important features.
2. Compare `max_depth=3` and unrestricted trees.
3. Change `n_estimators` from 50 to 500 and compare test performance.
4. Plot actual vs predicted for the random forest.
5. Find the 10 samples with the largest absolute errors.
6. Explain why random forests often generalize better than one deep tree.

## 7. Engineering problems

1. Why could age be important for concrete strength?
2. Why is feature importance useful to an engineer?
3. Why should a model not be used outside the range of mixtures represented in training data without caution?
4. What engineering checks should accompany an ML strength prediction?

## 8. Quiz

1. What is a decision-tree split?
2. What does max_depth control?
3. Define overfitting.
4. How does a random forest differ from a single tree?
5. Does feature importance prove causation?
6. Which metric would you use to express typical error in MPa?